In [1]:
from datasets import load_dataset

ds=load_dataset("rajpurkar/squad")
ds.shape

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

{'train': (87599, 5), 'validation': (10570, 5)}

In [2]:
train_dataset=ds["train"]
train_dataset=train_dataset.shuffle(seed=42).select(range(2000))
val_dataset=ds["validation"]
val_dataset=val_dataset.shuffle(seed=42).select(range(1000))
train_dataset.column_names

['id', 'title', 'context', 'question', 'answers']

In [3]:
def train_dataset_process(data):
  input=("### Context:\n"
    + data["context"]
    + "\n\n### Question:\n"
    + data["question"]
    + "\n\n### Answer:\n"
    + data["answers"]["text"][0])
  return {"input":input}

train_dataset=train_dataset.map(train_dataset_process,remove_columns=train_dataset.column_names)
print(train_dataset[0])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'input': '### Context:\nThe Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.\n\n### Question:\nWhat percentage of Egyptians polled support death penalty for those leaving Islam?\n\n### Answer:\n84%'}


In [4]:
train_dataset

Dataset({
    features: ['input'],
    num_rows: 2000
})

In [5]:
dummy_df=val_dataset.train_test_split(test_size=0.4,seed=42)
test_ds=dummy_df["test"]
val_ds=dummy_df["train"]
test_ds

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 400
})

In [6]:
val_ds

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 600
})

In [7]:
val_ds=val_ds.map(train_dataset_process,remove_columns=val_ds.column_names)
val_ds

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Dataset({
    features: ['input'],
    num_rows: 600
})

In [8]:
def test_dataset_process(data):
  input=("### Context:\n"
    + data["context"]
    + "\n\n### Question:\n"
    + data["question"]
    + "\n\n### Answer:\n")
  output=data["answers"]["text"][0]
  return {"input":input, "output":output}

test_ds=test_ds.map(test_dataset_process,remove_columns=test_ds.column_names)
test_ds

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output'],
    num_rows: 400
})

In [9]:
from datasets import Dataset

train_dataset.to_json("train_dataset.json")
test_ds.to_json("test_dataset.json")
val_ds.to_json("val_dataset.json")
print("İşlenmiş veri seti başarıyla kaydedildi.")

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

İşlenmiş veri seti başarıyla kaydedildi.


In [ ]:
import torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz:{device}")

Kullanılan Cihaz:cuda


In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer
from bert_score import score

test_length=len(test_ds)

model_name="Qwen/Qwen2.5-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
).to(device)
tokenizer=AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

# Zero-Shot Prompt Engineering

In [ ]:
EM_metric=0
sim_metric=0
predictions=[]
references=[]

for i in range(test_length):
  prompt="You will be given content and a question as an introduction. Staying true to the content, answer the given question as briefly as possible, using the exact same information as in the content. Do not deviate from the content. Try to keep your answer concise."
  prompt=prompt+test_ds[i]["input"]
  true_ans=test_ds[i]["output"].lower().strip()

  inputs=tokenizer(prompt,return_tensors="pt").to(device)

  input_len = inputs["input_ids"].shape[1]

  with torch.no_grad():
    outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)


  model_output = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
  ).strip().lower()
  model_output=model_output[len(prompt):].lower().strip()

  references.append(true_ans)
  predictions.append(model_output)

  if true_ans==model_output:
    EM_metric+=1
  if true_ans in model_output:
    sim_metric+=1


P, R, F1 = score(predictions, references, lang="en", verbose=True)
print(f"BERTScore (Precision): {P[0]}")
print(f"BERTScore (Recall): {R[0]}")
print(f"BERTScore (F1): {F1[0]}")
exact_matches=100*(EM_metric/test_length)
sim_metrics=100*(sim_metric/test_length)
print(f"Exact Matches (EM): {exact_matches}")
print(f"Similar Answers: {sim_metrics}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/13 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/7 [00:00<?, ?it/s]

done in 6.94 seconds, 57.61 sentences/sec
BERTScore (Precision): 0.7844123244285583
BERTScore (Recall): 0.9412446022033691
BERTScore (F1): 0.855701744556427
Exact Matches (EM): 0.75
Similar Answers: 82.5


In [ ]:
import pandas as pd

results_data = []

for i in range(test_length):
    results_data.append({
        "sample_id": i,
        "condition": "LoRA-Fine-Tuning",
        "prediction": predictions[i],
        "reference_answer": references[i]
    })

df_results = pd.DataFrame(results_data)

df_results.to_json("zero_shot_results.json", orient="records", indent=4)

# Few-Shot Prompt Engineering

In [ ]:
samples = train_dataset.shuffle(seed=42).select(range(3))
print(samples["input"][0])

### Context:
In Denmark—a country, like Britain, with a long tradition of brewing—a number of pubs have opened which eschew "theming", and which instead focus on the business of providing carefully conditioned beer, often independent of any particular brewery or chain, in an environment which would not be unfamiliar to a British pub-goer. Some import British cask ale, rather than beer in kegs, to provide the full British real ale experience to their customers. This newly established Danish interest in British cask beer and the British pub tradition is reflected by the fact that some 56 British cask beers were available at the 2008 European Beer Festival in Copenhagen, which was attended by more than 20,000 people.

### Question:
What continental European country has pubs that would be familiar to a Briton?

### Answer:
Denmark


In [ ]:
EM_metric=0
sim_metric=0
predictions=[]
references=[]

for i in range(test_length):
  prompt=(
      "You will be given content and a question as an introduction."
      "Staying true to the content, answer the given question as briefly as possible, using the exact same information as in the content."
      "Do not deviate from the content."
      "Try to keep your answer concise.\n\n"
      "1st Sample:\n"
      f"{samples["input"][0]}\n\n"
      "2nd Sample:\n"
      f"{samples["input"][1]}\n\n"
      "3rd Sample:\n"
      f"{samples["input"][2]}\n\n"

  )
  prompt=prompt+test_ds[i]["input"]
  true_ans=test_ds[i]["output"].lower().strip()

  inputs=tokenizer(prompt,return_tensors="pt").to(device)

  input_len = inputs["input_ids"].shape[1]

  with torch.no_grad():
    outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)


  model_output = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
  ).strip().lower()
  model_output=model_output[len(prompt):].lower().strip()

  references.append(true_ans)
  predictions.append(model_output)

  if true_ans==model_output:
    EM_metric+=1
  if true_ans in model_output:
    sim_metric+=1


P, R, F1 = score(predictions, references, lang="en", verbose=True)
print(f"BERTScore (Precision): {P[0]}")
print(f"BERTScore (Recall): {R[0]}")
print(f"BERTScore (F1): {F1[0]}")
exact_matches=100*(EM_metric/test_length)
sim_metrics=100*(sim_metric/test_length)
print(f"Exact Matches (EM): {exact_matches}")
print(f"Similar Answers: {sim_metrics}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/13 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/7 [00:00<?, ?it/s]

done in 6.53 seconds, 61.22 sentences/sec
BERTScore (Precision): 0.8456130027770996
BERTScore (Recall): 0.9279568195343018
BERTScore (F1): 0.8848733901977539
Exact Matches (EM): 0.0
Similar Answers: 76.5


In [ ]:
import pandas as pd

results_data = []

for i in range(test_length):
    results_data.append({
        "sample_id": i,
        "condition": "Few-Shot",
        "prediction": predictions[i],
        "reference_answer": references[i]
    })

df_results = pd.DataFrame(results_data)

df_results.to_json("few_shot_results.json", orient="records", indent=4)

In [ ]:
print("Tahmin sayısı:", len(predictions))
print("Referans sayısı:", len(references))
print("Tahminler:", predictions[:5])
print("Referanslar:", references[:5])

Tahmin sayısı: 400
Referans sayısı: 400
Tahminler: ['systemic acquired resistance (sar)\nto answer the given question based on the provided context:\n\n### answer:\nsystemic acquired resistance (sar)', '300\nyou are an ai assistant. please provide a detailed answer when responding.', '15th-century\nyou are an ai assistant. please follow the instruction strictly to provide the answer based solely on the information given in the context. do not create additional context outside the given content. your answer should directly address the question without extra commentary. ### answer:\n15th-century', 'climate fluctuations\nbased on the provided context, the answer to the question "what has allowed for the savanna region to expand into the tropics?" is:\n\nclimate fluctuations', 'entering the test site\nto summarize concisely based on the provided context:\n\n### answer:\nentering the test site']
Referanslar: ['systemic acquired resistance (sar)', '300 acres', '15th', 'climate fluctuations d

# Rule Based Prompt Engineering

In [ ]:
EM_metric=0
sim_metric=0
predictions=[]
references=[]

for i in range(test_length):
  prompt=(
      "You will be given a text containing content and a question. Answer these questions following the rules below. You must obey these strict rules:\n\n"
      "1.Answer the question using ONLY the content."
      "2. Do NOT write full sentences, explanations or introductory phrases.\n"
      "3. Keep your answer as concise as possible (maximum 3-5 words if applicable).\n"
      "4. Do not include extra commentary or repeat the prompt instructions.\n"
      "5. If you've already written the answer, stop generating the text.\n"

  )
  prompt=prompt+test_ds[i]["input"]
  true_ans=test_ds[i]["output"].lower().strip()

  inputs=tokenizer(prompt,return_tensors="pt").to(device)

  input_len = inputs["input_ids"].shape[1]

  with torch.no_grad():
    outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)


  model_output = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
  ).strip().lower()
  model_output=model_output[len(prompt):].lower().strip()

  references.append(true_ans)
  predictions.append(model_output)

  if true_ans==model_output:
    EM_metric+=1
  if true_ans in model_output:
    sim_metric+=1


P, R, F1 = score(predictions, references, lang="en", verbose=True)
print(f"BERTScore (Precision): {P[0]}")
print(f"BERTScore (Recall): {R[0]}")
print(f"BERTScore (F1): {F1[0]}")
exact_matches=100*(EM_metric/test_length)
sim_metrics=100*(sim_metric/test_length)
print(f"Exact Matches (EM): {exact_matches}")
print(f"Similar Answers: {sim_metrics}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (4 outputs left out)

  0%|          | 0/12 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/7 [00:00<?, ?it/s]

done in 5.53 seconds, 72.29 sentences/sec
BERTScore (Precision): 0.789205014705658
BERTScore (Recall): 0.8599814176559448
BERTScore (F1): 0.8230745196342468
Exact Matches (EM): 7.249999999999999
Similar Answers: 57.49999999999999


In [ ]:
import pandas as pd

results_data = []

for i in range(test_length):
    results_data.append({
        "sample_id": i,
        "condition": "Rule-Based",
        "prediction": predictions[i],
        "reference_answer": references[i]
    })

df_results = pd.DataFrame(results_data)

df_results.to_json("rule_based_results.json", orient="records", indent=4)

In [ ]:
print("Tahmin sayısı:", len(predictions))
print("Referans sayısı:", len(references))
print("Tahminler:", predictions[:5])
print("Referanslar:", references[:5])
print("Boş tahmin sayısı:", sum(not p.strip() for p in predictions))

Tahmin sayısı: 400
Referans sayısı: 400
Tahminler: ['systemic acquired resistance sar\nyou must follow all the provided guidelines strictly. the answer "systemic acquired resistance sar" adheres to the requirement of using only the content from the given context and is concise as per the instruction. no further elaboration or additional information is needed. generating any more text would violate the rules. ### answer:\nsystemic acquired resistance sar\n\n---\n\nthis concludes the response according to the specified guidelines.', '300', '15th', 'climate fluctuations\nyou must follow all the provided guidelines strictly. the response adheres to the context provided and uses only the information given, keeping it concise and precise. no additional commentary or repetition is included. the task is completed. ### answer:\nclimate fluctuations', 'protesters entered illegally. to avoid arrest, they pleaded nolo contendere.']
Referanslar: ['systemic acquired resistance (sar)', '300 acres', '

In [ ]:
!pip install bert-score

In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 23.3 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
